In [28]:
# ===== STEP 0: imports & globals =====
import os, glob
import pandas as pd

FOLDER = "rawdata/" #设置原始数据路径                 # change to "." if running inside the target folder
DATE_START, DATE_END = "2025-01-28", "2025-02-17" #设置预测的起始时间 和 结束时间
DATE_INDEX = pd.date_range(DATE_START, DATE_END, freq="D")
PRICE_COL = "平均价" # 从原始数据表中 选择平均价作为预测目标
FILE_GLOB = ['氧化钕.xls', '氧化铈.xls', '氧化镧.xls']

In [29]:
DATE_INDEX

DatetimeIndex(['2025-01-28', '2025-01-29', '2025-01-30', '2025-01-31',
               '2025-02-01', '2025-02-02', '2025-02-03', '2025-02-04',
               '2025-02-05', '2025-02-06', '2025-02-07', '2025-02-08',
               '2025-02-09', '2025-02-10', '2025-02-11', '2025-02-12',
               '2025-02-13', '2025-02-14', '2025-02-15', '2025-02-16',
               '2025-02-17'],
              dtype='datetime64[ns]', freq='D')

In [30]:
# ===== STEP 1: read each file, collect date spans, build raw =====
series_dict, spans = {}, {}
for i in FILE_GLOB:
    path = os.path.join(FOLDER, i)
    print(path)
    colname = os.path.splitext(os.path.basename(path))[0]
    
    # 读取文件，处理表头问题
    try:
        df = pd.read_excel(path, engine="xlrd")
        df.columns = df.columns.str.strip()
        if PRICE_COL not in df.columns:
            df = pd.read_excel(path, engine="xlrd", skiprows=1)
            df.columns = df.columns.str.strip()
            print(f"注意: {path} 已跳过第一行读取")
    except Exception as e:
        raise ValueError(f"读取 {path} 失败: {e}")
    
    if PRICE_COL not in df.columns:
        raise ValueError(f"{path} missing column {PRICE_COL!r}")
    
    # 清理数据：移除包含非日期信息的行
    date_col = df.columns[0]
    # 过滤掉包含特定关键词的行
    clean_mask = ~df[date_col].astype(str).str.contains(
        '来源：|数据来源|备注|注：|统计|整理', 
        case=False, na=False
    )
    df = df[clean_mask]
    
    # 使用更安全的日期转换
    s = (
        df.assign(date=pd.to_datetime(df.iloc[:, 0], errors='coerce'))
          .dropna(subset=['date'])  # 移除日期转换失败的行
          .set_index("date")[PRICE_COL]
          .sort_index()
          .rename(colname)
    )
    
    series_dict[colname] = s
    spans[colname] = (s.index.min(), s.index.max())

span_table = pd.DataFrame(spans, index=["start", "end"]).T
# display(span_table)

raw = pd.DataFrame(index=DATE_INDEX)
for name, s in series_dict.items():
    raw = raw.join(s, how="left")

rawdata/氧化钕.xls
注意: rawdata/氧化钕.xls 已跳过第一行读取
rawdata/氧化铈.xls
注意: rawdata/氧化铈.xls 已跳过第一行读取
rawdata/氧化镧.xls
注意: rawdata/氧化镧.xls 已跳过第一行读取


In [31]:

# ===== STEP 2: interpolate for late-start series -> test =====
merge_large = raw.copy()
for col in merge_large.columns:
    first_valid = merge_large[col].first_valid_index()
    if first_valid and first_valid > raw.index[0]:
        merge_large[col] = merge_large[col].interpolate("linear", limit_direction="both")
# 将处理好的数据放在XT_data文件夹下，命名为new_test.csv 作为测试数据
merge_large.to_csv("../XT_data/new_test1.csv", index_label="date", float_format="%.6f")
print("Saved new_test1.csv")

Saved new_test1.csv
